# Variable Clustering (VC) Analysis


In this section, I use **variable clustering** to detect groups of collinear or highly correlated variables. This technique helps reduce multicollinearity and select a more compact set of representative features, especially useful for scorecard models and interpretable ML.

The VarClusHi implementation uses **principal component analysis (PCA)** within clusters to retain meaningful structure while reducing redundancy.

In [2]:
!pip install factor_analyzer

import pandas as pd
import numpy as np
import statsmodels.api as sm

from varclus import VarClusHi

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Load Prepared Dataset

This dataset has already been pre-processed through WOE binning and univariate selection. It excludes the Information Value and loan_status columns for the purpose of correlation analysis.


In [3]:
data = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForUnivariateVSExceptIV.csv', index_col=[0])

In [4]:
data.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,1.002263,63788.0,17,-51.803563,5000.0,33.130018,11.01,0.08,12.0,686,-105.711715,0
11,1.002263,13113.0,0,131.015335,4500.0,-25.457321,8.63,0.34,2.0,651,-105.711715,1
9010,0.369066,59603.0,0,-51.803563,8000.0,11.545480,14.96,0.13,2.0,570,-105.711715,1
7030,-1.125529,54919.0,0,-51.803563,6225.0,55.131699,11.54,0.11,4.0,706,914.813057,0
21143,1.002263,55000.0,7,77.011839,6000.0,11.545480,9.32,0.11,9.0,643,914.813057,0


In [5]:
data.shape

(32400, 12)

## Drop Target Variable

loan_status is removed as it's the prediction target and should not influence clustering of input features.


In [6]:
data = data.drop(columns = ['loan_status'])

In [7]:
data.shape

(32400, 11)

In [8]:
datades = dict(data.describe())
[k for (k,v) in datades.items() if v['std'] == 0]

[]

## Run Variable Clustering


We run VarClusHi with:
- maxeigval2=0.7: maximum second eigenvalue threshold to stop splitting a cluster
- maxclus=None: allow the method to determine the optimal number of clusters

This approach identifies feature clusters based on similarity in variance structure.

In [9]:
data_vc = VarClusHi(data,maxeigval2=0.7,maxclus=None)
data_vc.varclus()

In [10]:
data_vc.info

,Cluster,N_Vars,Eigval1,Eigval2,VarProp
0,0,2,1.823416,0.176584,0.911708
1,1,2,1.596312,0.403688,0.798156
2,2,1,1.000000,0.000000,1.000000
3,3,1,1.000000,0.000000,1.000000
4,4,1,1.000000,0.000000,1.000000
5,5,1,1.000000,0.000000,1.000000
6,6,1,1.000000,0.000000,1.000000
7,7,1,1.000000,0.000000,1.000000
8,8,1,1.000000,0.000000,1.000000


In [11]:
test = data_vc.rsquare

## Interpretation of Variable Clustering Results

The table above shows the output of the Variable Clustering procedure:

- **Cluster**: Group of features with similar variance/correlation structure
- **RS_Own**: R² between a feature and its own cluster's first principal component (higher = better fit)
- **RS_NC**: R² between the feature and the nearest *other* cluster center (lower = better separation)
- **RS_Ratio**: Ratio of RS_NC to RS_Own — lower values indicate strong cluster membership

### Key Insights:
- Clusters 0 and 1 group features that are highly correlated:  
  - Cluster 0 includes person_emp_exp and cb_person_cred_hist_length, which are likely proxies for credit history length.  
  - Cluster 1 includes loan_amnt and loan_percent_income, both reflecting loan burden.

- All other features fall into **singleton clusters**, meaning they are not strongly collinear with others — a good sign for model stability.

- Features with **RS_Ratio close to 0** (e.g., credit_score, person_income) are **ideal for retention**, as they are both self-explanatory and statistically distinct.

> These insights will help drive dimensionality reduction by selecting **only one feature per cluster**, often the one with the **highest RS_Own** value or strongest business justification.


In [12]:
test

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio
0,0,person_emp_exp,0.911708,0.041047,9.207110e-02
1,0,cb_person_cred_hist_length,0.911708,0.024536,9.051265e-02
2,1,loan_amnt,0.798156,0.048339,2.120966e-01
3,1,loan_percent_income,0.798156,0.047749,2.119652e-01
4,2,previous_loan_defaults_on_file,1.000000,0.034715,0.000000e+00
5,3,credit_score,1.000000,0.033069,0.000000e+00
6,4,loan_intent,1.000000,0.006879,2.235827e-16
7,5,person_income,1.000000,0.029837,0.000000e+00
8,6,person_home_ownership,1.000000,0.027057,0.000000e+00
9,7,person_education,1.000000,0.015356,0.000000e+00


## Export Cluster Results


The output includes each feature’s:
- Assigned cluster
- R-squared with its own cluster center (within-cluster similarity)
- R-squared with the nearest other cluster (cross-cluster similarity)

This helps decide which variable to retain from each group (typically the one with highest within-cluster R²).

In [13]:
test.to_csv('Internal_VC_Results.csv')

In [14]:
test.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_VC_Results.csv')